# Acquisition des donnees -- JO 2028

---

## Contexte

Les Jeux Olympiques de Los Angeles 2028 approchent. Ce projet de **data storytelling** vise a
analyser l'histoire olympique a travers les donnees, de 1896 a 2024, pour en tirer des
perspectives sur l'edition 2028.

Ce premier notebook constitue la **phase d'acquisition**. Son objectif est triple :

1. **Inventorier** les sources de donnees disponibles et documenter leur provenance.
2. **Charger** chaque dataset et en dresser un premier portrait statistique.
3. **Verifier** la coherence entre les differentes sources avant de passer au nettoyage.

Aucune transformation n'est effectuee ici : on se contente d'observer les donnees brutes
telles qu'elles sont.

## Imports

In [1]:
# === Imports standard ===
import pandas as pd
import numpy as np
import os
import urllib.request  # MODIF : ajoute pour telecharger les CSV manquants

# === Configuration d'affichage ===
# Limites larges pour bien voir les datasets dans les cellules suivantes
pd.set_option("display.max_columns", 50)
pd.set_option("display.max_colwidth", 80)
pd.set_option("display.float_format", "{:.2f}".format)

# === Chemins des donnees ===
RAW_DIR = os.path.join("..", "data", "raw")
os.makedirs(RAW_DIR, exist_ok=True)  # MODIF : cree le dossier si absent (clone frais)

# === Telechargement automatique des CSV manquants ===
# MODIF (fix reproductibilite) : avant, les fichiers >10 Mo etaient gitignores
# mais aucune cellule ne les telechargeait -> un clone frais cassait le notebook.
# Source officielle : repo KeithGalli/Olympics-Dataset (scraping Olympedia.org),
# cite dans PLAN.md comme source de bios/bios_locs/results.
KEITHGALLI_BASE = "https://raw.githubusercontent.com/KeithGalli/Olympics-Dataset/master/clean-data"
FILES_TO_DOWNLOAD = {
    "bios.csv": f"{KEITHGALLI_BASE}/bios.csv",            # ~11 Mo, 145 500 athletes
    "bios_locs.csv": f"{KEITHGALLI_BASE}/bios_locs.csv",  # ~13 Mo, bios + lat/long
    "results.csv": f"{KEITHGALLI_BASE}/results.csv",      # ~33 Mo, 308 408 participations
}

# Telechargement conditionnel : skip si deja present (gain de temps sur re-runs)
for fname, url in FILES_TO_DOWNLOAD.items():
    fpath = os.path.join(RAW_DIR, fname)
    if not os.path.exists(fpath):
        print(f"Telechargement de {fname} depuis {url} ...")
        urllib.request.urlretrieve(url, fpath)
        size_mb = os.path.getsize(fpath) / (1024 * 1024)
        print(f"  OK -- {size_mb:.1f} Mo")
    else:
        size_mb = os.path.getsize(fpath) / (1024 * 1024)
        print(f"Deja present : {fname} ({size_mb:.1f} Mo)")

# Inventaire final pour verification visuelle
print("\nFichiers disponibles dans data/raw/ :")
for f in sorted(os.listdir(RAW_DIR)):
    size_mb = os.path.getsize(os.path.join(RAW_DIR, f)) / (1024 * 1024)
    print(f"  {f:30s} ({size_mb:.1f} Mo)")


Deja present : bios.csv (10.3 Mo)
Deja present : bios_locs.csv (12.0 Mo)
Deja present : results.csv (31.5 Mo)

Fichiers disponibles dans data/raw/ :
  bios.csv                       (10.3 Mo)
  bios_locs.csv                  (12.0 Mo)
  city_locations.csv             (0.0 Mo)
  noc_regions.csv                (0.0 Mo)
  olympics_1896_2024.csv         (2.2 Mo)
  populations.csv                (0.2 Mo)
  results.csv                    (31.5 Mo)


## Presentation des sources

Les donnees proviennent de repositories GitHub publics qui compilent des informations
historiques sur les Jeux Olympiques. Voici le detail de chaque source.

### Source 1 -- KeithGalli/Olympics-Dataset

| Champ       | Detail |
|-------------|--------|
| **URL**     | https://github.com/KeithGalli/Olympics-Dataset |
| **Fichiers** | `bios.csv`, `bios_locs.csv`, `results.csv` |
| **Description** | Dataset complet des athletes olympiques (biographies, localisations, resultats) scrape depuis les sources officielles olympiques. Couvre les JO d'ete et d'hiver de 1896 a 2024. |
| **Lignes attendues** | ~145 500 (bios), ~145 500 (bios_locs), ~308 400 (results) |
| **Licence** | MIT |

### Source 2 -- KeithGalli/Olympics-Dataset/clean-data

| Champ       | Detail |
|-------------|--------|
| **URL**     | https://github.com/KeithGalli/Olympics-Dataset/tree/main/clean-data |
| **Fichiers** | `noc_regions.csv`, `populations.csv` |
| **Description** | Donnees de reference : mapping des codes NOC vers les noms de pays/regions, et populations par pays de 1960 a 2023 (origine : Banque Mondiale). |
| **Lignes attendues** | ~229 (noc_regions), ~266 (populations) |
| **Licence** | MIT |

### Source 3 -- DjangoMustang/Olympics-1896-2024-Tableau

| Champ       | Detail |
|-------------|--------|
| **URL**     | https://github.com/DjangoMustang/Olympics-1896-2024-Tableau |
| **Fichiers** | `olympics_1896_2024.csv`, `city_locations.csv` |
| **Description** | Dataset consolide des medailles olympiques de 1896 a 2024 (incluant Paris 2024), accompagne des coordonnees geographiques des villes hotes. Concu pour la visualisation Tableau. |
| **Lignes attendues** | ~24 000 (olympics_1896_2024), ~55 (city_locations) |
| **Licence** | Libre (GitHub public) |

---

## Chargement des datasets

On charge chaque fichier CSV et on realise un premier examen : dimensions, types de colonnes,
apercu des premieres lignes et statistiques descriptives.

### 1. `bios.csv` -- Biographies des athletes

Ce fichier contient les informations biographiques de tous les athletes olympiques.
Chaque ligne represente un athlete unique identifie par `athlete_id`. On y retrouve
le nom complet, le sexe, le pays (NOC), les dates de naissance/deces, et des
informations complementaires (mensurations, affiliations, surnoms).

In [2]:
bios = pd.read_csv(os.path.join(RAW_DIR, "bios.csv"))

print(f"Dimensions : {bios.shape[0]:,} lignes x {bios.shape[1]} colonnes")
print(f"\nTypes des colonnes :")
print(bios.dtypes)
print(f"\nApercu des premieres lignes :")
bios.head()

Dimensions : 145,500 lignes x 10 colonnes

Types des colonnes :
athlete_id        int64
name                str
born_date           str
born_city           str
born_region         str
born_country        str
NOC                 str
height_cm       float64
weight_kg       float64
died_date           str
dtype: object

Apercu des premieres lignes :


,athlete_id,name,born_date,born_city,born_region,born_country,NOC,height_cm,weight_kg,died_date
0,1,Jean-François Blanchy,1886-12-12,Bordeaux,Gironde,FRA,France,NaN,NaN,1960-10-02
1,2,Arnaud Boetsch,1969-04-01,Meulan,Yvelines,FRA,France,183.00,76.00,NaN
2,3,Jean Borotra,1898-08-13,Biarritz,Pyrénées-Atlantiques,FRA,France,183.00,76.00,1994-07-17
3,4,Jacques Brugnon,1895-05-11,Paris VIIIe,Paris,FRA,France,168.00,64.00,1978-03-20
4,5,Albert Canet,1878-04-17,Wandsworth,England,GBR,France,NaN,NaN,1930-07-25


In [3]:
print("Statistiques descriptives (colonnes numeriques) :")
bios.describe()

Statistiques descriptives (colonnes numeriques) :


,athlete_id,height_cm,weight_kg
count,145500.00,106651.00,102070.00
mean,73686.19,176.33,71.89
std,42868.96,10.38,14.47
min,1.00,127.00,25.00
25%,36663.75,170.00,62.00
50%,73302.50,176.00,70.00
75%,110306.25,183.00,80.00
max,149814.00,226.00,198.00


In [4]:
print("Valeurs manquantes par colonne :")
missing = bios.isnull().sum()
missing_pct = (missing / len(bios) * 100).round(1)
pd.DataFrame({"manquantes": missing, "pourcentage": missing_pct}).query("manquantes > 0").sort_values("pourcentage", ascending=False)

Valeurs manquantes par colonne :


,manquantes,pourcentage
died_date,111560,76.70
weight_kg,43430,29.80
height_cm,38849,26.70
born_city,34592,23.80
born_region,34592,23.80
born_country,34592,23.80
born_date,1807,1.20
NOC,1,0.00


**Observations sur `bios.csv` :**

- Le fichier contient environ 145 500 athletes, ce qui couvre l'ensemble de l'histoire olympique.
- Plusieurs colonnes presentent un fort taux de valeurs manquantes (`Died`, `Nick/petnames`, `Title(s)`, `Original name`) -- ce qui est attendu pour des champs optionnels.
- Les colonnes cles pour nos analyses seront : `athlete_id`, `Sex`, `NOC`, `Born`, `Measurements`.
- La colonne `Born` est en format texte (ex: "12 December 1886 in Bordeaux...") et necessiterait un parsing lors du nettoyage.

### 2. `bios_locs.csv` -- Biographies avec geolocalisation

Version structuree des biographies, avec les informations de naissance deja parsees
(ville, region, pays) et les coordonnees geographiques (latitude/longitude).
Contient aussi la taille et le poids extraits de la colonne `Measurements` du fichier precedent.

In [5]:
bios_locs = pd.read_csv(os.path.join(RAW_DIR, "bios_locs.csv"))

print(f"Dimensions : {bios_locs.shape[0]:,} lignes x {bios_locs.shape[1]} colonnes")
print(f"\nTypes des colonnes :")
print(bios_locs.dtypes)
print(f"\nApercu des premieres lignes :")
bios_locs.head()

Dimensions : 145,500 lignes x 12 colonnes

Types des colonnes :
athlete_id        int64
name                str
born_date           str
born_city           str
born_region         str
born_country        str
NOC                 str
height_cm       float64
weight_kg       float64
died_date           str
lat             float64
long            float64
dtype: object

Apercu des premieres lignes :


,athlete_id,name,born_date,born_city,born_region,born_country,NOC,height_cm,weight_kg,died_date,lat,long
0,1,Jean-François Blanchy,1886-12-12,Bordeaux,Gironde,FRA,France,NaN,NaN,1960-10-02,44.84,-0.58
1,2,Arnaud Boetsch,1969-04-01,Meulan,Yvelines,FRA,France,183.00,76.00,NaN,49.00,1.91
2,3,Jean Borotra,1898-08-13,Biarritz,Pyrénées-Atlantiques,FRA,France,183.00,76.00,1994-07-17,43.47,-1.55
3,4,Jacques Brugnon,1895-05-11,Paris VIIIe,Paris,FRA,France,168.00,64.00,1978-03-20,48.88,2.32
4,5,Albert Canet,1878-04-17,Wandsworth,England,GBR,France,NaN,NaN,1930-07-25,51.46,-0.19


In [6]:
print("Statistiques descriptives :")
bios_locs.describe()

Statistiques descriptives :


,athlete_id,height_cm,weight_kg,lat,long
count,145500.00,106651.00,102070.00,69545.00,69545.00
mean,73686.19,176.33,71.89,38.94,0.23
std,42868.96,10.38,14.47,20.82,49.22
min,1.00,127.00,25.00,-53.16,-178.81
25%,36663.75,170.00,62.00,40.42,-3.71
50%,73302.50,176.00,70.00,41.88,-0.35
75%,110306.25,183.00,80.00,50.26,14.45
max,149814.00,226.00,198.00,74.98,179.77


In [7]:
print("Valeurs manquantes par colonne :")
missing = bios_locs.isnull().sum()
missing_pct = (missing / len(bios_locs) * 100).round(1)
pd.DataFrame({"manquantes": missing, "pourcentage": missing_pct}).query("manquantes > 0").sort_values("pourcentage", ascending=False)

Valeurs manquantes par colonne :


,manquantes,pourcentage
died_date,111560,76.70
lat,75955,52.20
long,75955,52.20
weight_kg,43430,29.80
height_cm,38849,26.70
born_city,34592,23.80
born_region,34592,23.80
born_country,34592,23.80
born_date,1807,1.20
NOC,1,0.00


**Observations sur `bios_locs.csv` :**

- Meme nombre de lignes que `bios.csv` : il y a bien une correspondance 1:1 via `athlete_id`.
- Les colonnes `height_cm` et `weight_kg` sont numeriques et pretes a l'emploi, mais presentent beaucoup de valeurs manquantes (mensurations non disponibles pour les athletes historiques).
- Les coordonnees `lat`/`long` permettront des visualisations cartographiques sur l'origine geographique des athletes.
- La colonne `born_date` est au format ISO (YYYY-MM-DD), beaucoup plus exploitable que le champ `Born` de `bios.csv`.

### 3. `results.csv` -- Resultats olympiques

Le coeur du dataset : chaque ligne represente la participation d'un athlete a une epreuve
lors d'une edition des Jeux. On y retrouve le classement (`Pos`), la medaille eventuelle,
la discipline et l'epreuve.

In [8]:
results = pd.read_csv(os.path.join(RAW_DIR, "results.csv"))

print(f"Dimensions : {results.shape[0]:,} lignes x {results.shape[1]} colonnes")
print(f"\nTypes des colonnes :")
print(results.dtypes)
print(f"\nApercu des premieres lignes :")
results.head()

Dimensions : 308,408 lignes x 11 colonnes

Types des colonnes :
year          float64
type              str
discipline        str
event             str
as                str
athlete_id      int64
noc               str
team              str
place         float64
tied             bool
medal             str
dtype: object

Apercu des premieres lignes :


,year,type,discipline,event,as,athlete_id,noc,team,place,tied,medal
0,1912.00,Summer,Tennis,"Singles, Men (Olympic)",Jean-François Blanchy,1,FRA,NaN,17.00,True,NaN
1,1912.00,Summer,Tennis,"Doubles, Men (Olympic)",Jean-François Blanchy,1,FRA,Jean Montariol,NaN,False,NaN
2,1920.00,Summer,Tennis,"Singles, Men (Olympic)",Jean-François Blanchy,1,FRA,NaN,32.00,True,NaN
3,1920.00,Summer,Tennis,"Doubles, Mixed (Olympic)",Jean-François Blanchy,1,FRA,Jeanne Vaussard,8.00,True,NaN
4,1920.00,Summer,Tennis,"Doubles, Men (Olympic)",Jean-François Blanchy,1,FRA,Jacques Brugnon,4.00,False,NaN


In [9]:
print("Statistiques descriptives :")
results.describe(include="all")

Statistiques descriptives :


,year,type,discipline,event,as,athlete_id,noc,team,place,tied,medal
count,305807.00,305807,308407,308408,308408,308408.00,308407,121714,283193.00,308408,44139
unique,NaN,2,93,1191,144005,NaN,230,19413,NaN,2,3
top,NaN,Summer,Athletics,"Football, Men (Olympic)",Heikki Savolainen,NaN,USA,United States,NaN,False,Bronze
freq,NaN,241298,45916,6135,39,NaN,21353,6149,NaN,262468,14810
mean,1982.24,NaN,NaN,NaN,NaN,72624.10,NaN,NaN,16.10,NaN,NaN
std,30.98,NaN,NaN,NaN,NaN,41285.94,NaN,NaN,19.32,NaN,NaN
min,1896.00,NaN,NaN,NaN,NaN,1.00,NaN,NaN,1.00,NaN,NaN
25%,1964.00,NaN,NaN,NaN,NaN,34412.00,NaN,NaN,5.00,NaN,NaN
50%,1992.00,NaN,NaN,NaN,NaN,73739.50,NaN,NaN,9.00,NaN,NaN
75%,2008.00,NaN,NaN,NaN,NaN,107168.00,NaN,NaN,20.00,NaN,NaN


In [10]:
print("Valeurs manquantes par colonne :")
missing = results.isnull().sum()
missing_pct = (missing / len(results) * 100).round(1)
pd.DataFrame({"manquantes": missing, "pourcentage": missing_pct}).query("manquantes > 0").sort_values("pourcentage", ascending=False)

Valeurs manquantes par colonne :


,manquantes,pourcentage
medal,264269,85.70
team,186694,60.50
place,25215,8.20
year,2601,0.80
type,2601,0.80
discipline,1,0.00
noc,1,0.00


In [11]:
# === Statistiques sur les medailles ===
# MODIF : "Medal" -> "medal", "Games" -> "year", "Discipline" -> "discipline"
# Le dataset KeithGalli utilise des noms en minuscules (cf. cellule 16 dtypes).
# L'ancien code (PascalCase) etait herite du dataset Kaggle Rgriffin.
print("Distribution des medailles :")
print(results["medal"].value_counts(dropna=False))

# Editions distinctes = nombre d'annees distinctes (pas de colonne 'Games' dans KeithGalli)
print(f"\nNombre d'editions distinctes (par annee) : {results['year'].nunique()}")
print(f"Nombre de disciplines distinctes : {results['discipline'].nunique()}")


Distribution des medailles :
medal
NaN       264269
Bronze     14810
Gold       14783
Silver     14546
Name: count, dtype: int64

Nombre d'editions distinctes (par annee) : 37
Nombre de disciplines distinctes : 93


**Observations sur `results.csv` :**

- Plus de 308 000 participations enregistrees, couvrant toutes les editions.
- La colonne `Medal` est majoritairement vide (NaN) : normal, seuls les medailles sont renseignes.
- La colonne `Unnamed: 7` est entierement vide -- artefact d'export a ignorer.
- La colonne `Games` contient le format "YYYY Season Olympics" (ex: "2024 Summer Olympics"), ce qui permettra d'extraire l'annee et la saison.
- Le lien avec les biographies se fait via `athlete_id`.

### 4. `olympics_1896_2024.csv` -- Medailles 1896-2024

Dataset complementaire focalise uniquement sur les medaillees. Plus compact que `results.csv`,
il a l'avantage d'inclure les donnees de Paris 2024 dans un format propre et de contenir
le nom de la ville hote et l'annee directement en colonnes.

In [12]:
olympics = pd.read_csv(os.path.join(RAW_DIR, "olympics_1896_2024.csv"))

print(f"Dimensions : {olympics.shape[0]:,} lignes x {olympics.shape[1]} colonnes")
print(f"\nTypes des colonnes :")
print(olympics.dtypes)
print(f"\nApercu des premieres lignes :")
olympics.head()

Dimensions : 24,012 lignes x 10 colonnes

Types des colonnes :
discipline        str
event             str
event_gender      str
event_type        str
medal_type        str
name              str
country_code      str
country           str
city              str
event_year      int64
dtype: object

Apercu des premieres lignes :


,discipline,event,event_gender,event_type,medal_type,name,country_code,country,city,event_year
0,Cycling Road,Men Individual Time Trial,Male,Athlete,Gold,Evenepoel Remco,BEL,Belgium,Paris,2024
1,Cycling Road,Men Individual Time Trial,Male,Athlete,Silver,Ganna Filippo,ITA,Italy,Paris,2024
2,Cycling Road,Men Individual Time Trial,Male,Athlete,Bronze,Van Aert Wout,BEL,Belgium,Paris,2024
3,Cycling Road,Women Individual Time Trial,Female,Athlete,Gold,Brown Grace,AUS,Australia,Paris,2024
4,Cycling Road,Women Individual Time Trial,Female,Athlete,Silver,Henderson Anna,GBR,Great Britain,Paris,2024


In [13]:
print("Statistiques descriptives :")
olympics.describe(include="all")

Statistiques descriptives :


,discipline,event,event_gender,event_type,medal_type,name,country_code,country,city,event_year
count,24012,24012,24012,24012,24012,20388,24012,24012,24012,24012.00
unique,87,1310,3,2,3,14934,160,162,43,NaN
top,Athletics,Men,Male,Athlete,Bronze,Michael Phelps,USA,United States of America,Paris,NaN
freq,3311,399,15173,15873,8336,16,3424,3424,3002,NaN
mean,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,1984.26
std,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,34.26
min,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,1896.00
25%,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,1964.00
50%,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,1992.00
75%,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,2012.00


In [14]:
print("Valeurs manquantes par colonne :")
missing = olympics.isnull().sum()
missing_pct = (missing / len(olympics) * 100).round(1)
pd.DataFrame({"manquantes": missing, "pourcentage": missing_pct}).query("manquantes > 0").sort_values("pourcentage", ascending=False)

Valeurs manquantes par colonne :


,manquantes,pourcentage
name,3624,15.10


In [15]:
print("Repartition par type de medaille :")
print(olympics["medal_type"].value_counts())

print(f"\nPeriode couverte : {olympics['event_year'].min()} - {olympics['event_year'].max()}")
print(f"Nombre de villes hotes : {olympics['city'].nunique()}")
print(f"\nVilles hotes :")
print(olympics.groupby("city")["event_year"].apply(lambda x: sorted(x.unique())).to_string())

Repartition par type de medaille :
medal_type
Bronze    8336
Gold      7861
Silver    7815
Name: count, dtype: int64

Periode couverte : 1896 - 2024
Nombre de villes hotes : 43

Villes hotes :
city
Albertville                           [1992]
Amsterdam                             [1928]
Antwerp                               [1920]
Athens                          [1896, 2004]
Atlanta                               [1996]
Barcelona                             [1992]
Beijing                         [2008, 2022]
Berlin                                [1936]
Calgary                               [1988]
Chamonix                              [1924]
Cortina D Ampezzo                     [1956]
Garmisch Partenkirchen                [1936]
Grenoble                              [1968]
Helsinki                              [1952]
Innsbruck                       [1964, 1976]
Lake Placid                     [1932, 1980]
Lillehammer                           [1994]
London                    [1908, 1948

**Observations sur `olympics_1896_2024.csv` :**

- Environ 24 000 medailles enregistrees, de 1896 a 2024.
- Format propre avec `event_year` et `city` directement exploitables.
- Contient les colonnes `event_gender` et `event_type` qui permettent de distinguer les epreuves individuelles des epreuves par equipe.
- Ce dataset sera particulierement utile pour les analyses de tendances sur les medailles.

### 5. `noc_regions.csv` -- Mapping NOC / regions

Table de reference qui associe chaque code NOC (National Olympic Committee) a un nom
de pays ou de region. Indispensable pour les jointures et les visualisations par pays.

In [16]:
noc_regions = pd.read_csv(os.path.join(RAW_DIR, "noc_regions.csv"))

print(f"Dimensions : {noc_regions.shape[0]:,} lignes x {noc_regions.shape[1]} colonnes")
print(f"\nTypes des colonnes :")
print(noc_regions.dtypes)
print(f"\nApercu :")
noc_regions.head(10)

Dimensions : 230 lignes x 3 colonnes

Types des colonnes :
NOC       str
region    str
notes     str
dtype: object

Apercu :


,NOC,region,notes
0,AFG,Afghanistan,NaN
1,AHO,Curacao,Netherlands Antilles
2,ALB,Albania,NaN
3,ALG,Algeria,NaN
4,AND,Andorra,NaN
5,ANG,Angola,NaN
6,ANT,Antigua,Antigua and Barbuda
7,ANZ,Australia,Australasia
8,ARG,Argentina,NaN
9,ARM,Armenia,NaN


In [17]:
print("Valeurs manquantes :")
print(noc_regions.isnull().sum())
print(f"\nNombre de codes NOC uniques : {noc_regions['NOC'].nunique()}")
print(f"Doublons sur NOC : {noc_regions['NOC'].duplicated().sum()}")

Valeurs manquantes :
NOC         0
region      3
notes     209
dtype: int64

Nombre de codes NOC uniques : 230
Doublons sur NOC : 0


**Observations sur `noc_regions.csv` :**

- 229 codes NOC references, ce qui couvre largement les comites olympiques actuels et historiques.
- La colonne `notes` est souvent vide : elle ne contient des precisions que pour les cas particuliers (pays disparus, renommes, etc.).
- Servira de table de jointure pour normaliser les noms de pays a travers les datasets.

### 6. `populations.csv` -- Population par pays (1960-2023)

Donnees de population issues de la Banque Mondiale. Chaque ligne correspond a un pays,
avec une colonne par annee de 1960 a 2023. Permettra de normaliser les performances
olympiques par habitant.

In [18]:
populations = pd.read_csv(os.path.join(RAW_DIR, "populations.csv"))

print(f"Dimensions : {populations.shape[0]:,} lignes x {populations.shape[1]} colonnes")
print(f"\nColonnes (2 premieres + 5 dernieres annees) :")
print(f"  Identifiants : {list(populations.columns[:2])}")
print(f"  Annees : {list(populations.columns[2:5])} ... {list(populations.columns[-3:])}")
print(f"\nApercu :")
populations.head()

Dimensions : 266 lignes x 66 colonnes

Colonnes (2 premieres + 5 dernieres annees) :
  Identifiants : ['Country Name', 'Country Code']
  Annees : ['1960', '1961', '1962'] ... ['2021', '2022', '2023']

Apercu :


,Country Name,Country Code,1960,1961,1962,1963,1964,1965,1966,1967,1968,1969,1970,1971,1972,1973,1974,1975,1976,1977,1978,1979,1980,1981,1982,...,1999,2000,2001,2002,2003,2004,2005,2006,2007,2008,2009,2010,2011,2012,2013,2014,2015,2016,2017,2018,2019,2020,2021,2022,2023
0,Aruba,ABW,54608.00,55811.00,56682.00,57475.00,58178.00,58782.00,59291.00,59522.00,59471.00,59330.00,59106.00,58816.00,58855.00,59365.00,60028.00,60715.00,61193.00,61465.00,61738.00,62006.00,62267.00,62614.00,63116.00,...,86867.00,89101.00,90691.00,91781.00,92701.00,93540.00,94483.00,95606.00,96787.00,97996.00,99212.00,100341.00,101288.00,102112.00,102880.00,103594.00,104257.00,104874.00,105439.00,105962.00,106442.00,106585.00,106537.00,106445.00,106277.00
1,Africa Eastern and Southern,AFE,130692579.00,134169237.00,137835590.00,141630546.00,145605995.00,149742351.00,153955516.00,158313235.00,162875171.00,167596160.00,172475766.00,177503186.00,182599092.00,187901657.00,193512956.00,199284304.00,205202669.00,211120911.00,217481420.00,224315978.00,230967858.00,237937461.00,245386717.00,...,391486231.00,401600588.00,412001885.00,422741118.00,433807484.00,445281555.00,457153837.00,469508516.00,482406426.00,495748900.00,509410477.00,523459657.00,537792950.00,552530654.00,567892149.00,583651101.00,600008424.00,616377605.00,632746570.00,649757148.00,667242986.00,685112979.00,702977106.00,720859132.00,739108306.00
2,Afghanistan,AFG,8622466.00,8790140.00,8969047.00,9157465.00,9355514.00,9565147.00,9783147.00,10010030.00,10247780.00,10494489.00,10752971.00,11015857.00,11286753.00,11575305.00,11869879.00,12157386.00,12425267.00,12687301.00,12938862.00,12986369.00,12486631.00,11155195.00,10088289.00,...,19262847.00,19542982.00,19688632.00,21000256.00,22645130.00,23553551.00,24411191.00,25442944.00,25903301.00,26427199.00,27385307.00,28189672.00,29249157.00,30466479.00,31541209.00,32716210.00,33753499.00,34636207.00,35643418.00,36686784.00,37769499.00,38972230.00,40099462.00,41128771.00,42239854.00
3,Africa Western and Central,AFW,97256290.00,99314028.00,101445032.00,103667517.00,105959979.00,108336203.00,110798486.00,113319950.00,115921723.00,118615741.00,121424797.00,124336039.00,127364044.00,130563107.00,133953892.00,137548613.00,141258400.00,145122851.00,149206663.00,153459665.00,157825609.00,162323313.00,167023385.00,...,262397030.00,269611898.00,277160097.00,284952322.00,292977949.00,301265247.00,309824829.00,318601484.00,327612838.00,336893835.00,346475221.00,356337762.00,366489204.00,376797999.00,387204553.00,397855507.00,408690375.00,419778384.00,431138704.00,442646825.00,454306063.00,466189102.00,478185907.00,490330870.00,502789511.00
4,Angola,AGO,5357195.00,5441333.00,5521400.00,5599827.00,5673199.00,5736582.00,5787044.00,5827503.00,5868203.00,5928386.00,6029700.00,6177049.00,6364731.00,6578230.00,6802494.00,7032713.00,7266780.00,7511895.00,7771590.00,8043218.00,8330047.00,8631457.00,8947152.00,...,15870753.00,16394062.00,16941587.00,17516139.00,18124342.00,18771125.00,19450959.00,20162340.00,20909684.00,21691522.00,22507674.00,23364185.00,24259111.00,25188292.00,26147002.00,27128337.00,28127721.00,29154746.00,30208628.00,31273533.00,32353588.00,33428486.00,34503774.00,35588987.00,36684202.00


In [19]:
print(f"Nombre de pays/entites : {len(populations)}")
print(f"Periode couverte : 1960 - 2023")

# Taux de valeurs manquantes sur les dernieres annees
year_cols = [str(y) for y in range(2019, 2024)]
for col in year_cols:
    if col in populations.columns:
        pct = populations[col].isnull().mean() * 100
        print(f"  Manquantes en {col} : {pct:.1f}%")

Nombre de pays/entites : 266
Periode couverte : 1960 - 2023
  Manquantes en 2019 : 0.4%
  Manquantes en 2020 : 0.4%
  Manquantes en 2021 : 0.4%
  Manquantes en 2022 : 0.4%
  Manquantes en 2023 : 0.4%


**Observations sur `populations.csv` :**

- 266 entites (pays + regions agregees comme "Africa Eastern and Southern").
- Le `Country Code` utilise le format ISO 3166-1 alpha-3, qui differe parfois des codes NOC. Une table de correspondance sera necessaire.
- Les populations recentes (2020-2023) sont bien renseignees pour la plupart des pays.

### 7. `city_locations.csv` -- Coordonnees des villes hotes

Petit fichier de reference qui associe chaque edition des JO a la ville hote
et ses coordonnees geographiques. Utile pour les cartes.

In [20]:
city_locations = pd.read_csv(os.path.join(RAW_DIR, "city_locations.csv"))

print(f"Dimensions : {city_locations.shape[0]:,} lignes x {city_locations.shape[1]} colonnes")
print(f"\nContenu complet :")
city_locations

Dimensions : 55 lignes x 4 colonnes

Contenu complet :


,Year,City,latitude,longitude
0,1896,Athens,37.98,23.73
1,1900,Paris,48.86,2.35
2,1904,St Louis,38.63,-90.20
3,1908,London,51.51,-0.13
4,1912,Stockholm,59.33,18.07
5,1916,Berlin,52.52,13.41
6,1920,Antwerp,51.22,4.40
7,1924,Chamonix,45.55,6.52
8,1924,Paris,48.86,2.35
9,1928,Amsterdam,52.37,4.89


**Observations sur `city_locations.csv` :**

- 55 entrees couvrant toutes les editions des JO d'ete et d'hiver.
- Certaines villes apparaissent plusieurs fois (Paris : 1900, 1924, 2024 ; Londres : 1908, 1948, 2012).
- Les coordonnees sont en format decimal, directement exploitables pour la cartographie.

---

## Vue d'ensemble

Maintenant que chaque dataset a ete charge individuellement, construisons un tableau
recapitulatif pour avoir une vision globale de notre corpus de donnees.

In [21]:
datasets = {
    "bios": bios,
    "bios_locs": bios_locs,
    "results": results,
    "olympics_1896_2024": olympics,
    "noc_regions": noc_regions,
    "populations": populations,
    "city_locations": city_locations,
}

summary_rows = []
for name, df in datasets.items():
    n_rows, n_cols = df.shape
    nan_pct = (df.isnull().sum().sum() / (n_rows * n_cols) * 100)
    mem_mb = df.memory_usage(deep=True).sum() / (1024 * 1024)
    summary_rows.append({
        "Dataset": name,
        "Lignes": f"{n_rows:,}",
        "Colonnes": n_cols,
        "NaN (%)": f"{nan_pct:.1f}%",
        "Memoire (Mo)": f"{mem_mb:.1f}",
        "Colonnes (liste)": ", ".join(df.columns[:6].tolist()) + ("..." if n_cols > 6 else ""),
    })

summary_df = pd.DataFrame(summary_rows)
summary_df

,Dataset,Lignes,Colonnes,NaN (%),Memoire (Mo),Colonnes (liste)
0,bios,"145,500",10,20.6%,18.5,"athlete_id, name, born_date, born_city, born_region, born_country..."
1,bios_locs,"145,500",12,25.8%,20.7,"athlete_id, name, born_date, born_city, born_region, born_country..."
2,results,"308,408",11,14.2%,46.0,"year, type, discipline, event, as, athlete_id..."
3,olympics_1896_2024,"24,012",10,1.5%,3.7,"discipline, event, event_gender, event_type, medal_type, name..."
4,noc_regions,230,3,30.7%,0.0,"NOC, region, notes"
5,populations,266,66,0.5%,0.1,"Country Name, Country Code, 1960, 1961, 1962, 1963..."
6,city_locations,55,4,0.0%,0.0,"Year, City, latitude, longitude"


Le corpus total represente environ 750 000 lignes reparties sur 7 fichiers. Les deux
fichiers principaux (`bios` et `results`) constituent l'essentiel du volume. Les fichiers
de reference (`noc_regions`, `populations`, `city_locations`) sont legers mais indispensables
pour enrichir les analyses.

---

## Verification de coherence

Avant de passer au nettoyage, verifions que les datasets sont coherents entre eux
et qu'aucune donnee critique ne manque.

### Athletes : `bios` vs `results`

L'`athlete_id` est la cle de jointure entre les biographies et les resultats.
Verifions la correspondance.

In [22]:
bios_ids = set(bios["athlete_id"].dropna().unique())
results_ids = set(results["athlete_id"].dropna().unique())
bios_locs_ids = set(bios_locs["athlete_id"].dropna().unique())

print(f"Athletes dans bios :        {len(bios_ids):,}")
print(f"Athletes dans bios_locs :    {len(bios_locs_ids):,}")
print(f"Athletes dans results :      {len(results_ids):,}")

print(f"\n-- Intersections --")
print(f"Dans bios ET results :       {len(bios_ids & results_ids):,}")
print(f"Dans bios MAIS PAS results : {len(bios_ids - results_ids):,}")
print(f"Dans results MAIS PAS bios : {len(results_ids - bios_ids):,}")

print(f"\n-- Correspondance bios / bios_locs --")
print(f"Identiques : {bios_ids == bios_locs_ids}")

Athletes dans bios :        145,500
Athletes dans bios_locs :    145,500
Athletes dans results :      145,262

-- Intersections --
Dans bios ET results :       145,262
Dans bios MAIS PAS results : 238
Dans results MAIS PAS bios : 0

-- Correspondance bios / bios_locs --
Identiques : True


### Codes NOC : `results` vs `noc_regions`

Verifions que tous les codes NOC presents dans les resultats sont bien references
dans la table de mapping.

In [23]:
# === Verification : tous les NOC de results sont-ils dans noc_regions ? ===
# MODIF : "NOC" -> "noc" dans le DataFrame results (KeithGalli minuscules).
# Attention : la colonne NOC du fichier noc_regions.csv reste en majuscules.
# str.upper() applique pour matcher (resilience contre majuscule/minuscule).
results_nocs = set(results["noc"].dropna().str.upper().unique())
ref_nocs = set(noc_regions["NOC"].dropna().unique())

# Difference = NOC presents dans les resultats mais sans entree de mapping
# -> ils n'auront pas de nom de pays associe lors du merge en phase 3
missing_nocs = results_nocs - ref_nocs

print(f"Codes NOC dans results :     {len(results_nocs)}")
print(f"Codes NOC dans noc_regions : {len(ref_nocs)}")
print(f"\nNOC dans results MAIS PAS dans noc_regions ({len(missing_nocs)}) :")

if missing_nocs:
    for noc in sorted(missing_nocs):
        # Compter les participations affectees pour mesurer la severite du gap
        count = results[results["noc"].str.upper() == noc].shape[0]
        print(f"  {noc} ({count} participations)")
else:
    print("  Aucun -- tous les NOC sont references.")


Codes NOC dans results :     230
Codes NOC dans noc_regions : 230

NOC dans results MAIS PAS dans noc_regions (5) :


  COR (26 participations)


  EOR (47 participations)
  LBN (377 participations)
  ROC (1128 participations)
  SGP (397 participations)


### Couverture temporelle

Verifions les periodes couvertes par chaque source et assurons-nous que
les donnees les plus recentes (Paris 2024) sont bien presentes.

In [24]:
# === Couverture temporelle des datasets ===
# MODIF : la colonne 'year' est native dans results.csv (KeithGalli),
# plus besoin de la reconstruire depuis 'Games' comme dans l'ancien code Kaggle.
# On affiche aussi 'type' (Summer/Winter) qui est specifique au schema KeithGalli.
print("=== Couverture temporelle ===")
print(f"\nresults.csv :")
print(f"  Annees : {int(results['year'].min())} - {int(results['year'].max())}")
print(f"  Editions : {results['year'].nunique()}")
print(f"  Saisons (Summer/Winter) : {results['type'].value_counts(dropna=False).to_dict()}")

# olympics_1896_2024 : couvre Paris 2024, complementaire a results (qui s'arrete a 2022)
print(f"\nolympics_1896_2024.csv :")
print(f"  Annees : {olympics['event_year'].min()} - {olympics['event_year'].max()}")

# city_locations : coordonnees des villes hotes (utilise dans la phase nettoyage)
print(f"\ncity_locations.csv :")
print(f"  Annees : {city_locations['Year'].min()} - {city_locations['Year'].max()}")

# populations : World Bank, colonnes-annees a partir de 1960
print(f"\npopulations.csv :")
year_cols_pop = [c for c in populations.columns if c.isdigit()]
print(f"  Annees : {min(year_cols_pop)} - {max(year_cols_pop)}")


=== Couverture temporelle ===

results.csv :
  Annees : 1896 - 2022
  Editions : 37
  Saisons (Summer/Winter) : {'Summer': 241298, 'Winter': 64509, nan: 2601}

olympics_1896_2024.csv :
  Annees : 1896 - 2024

city_locations.csv :
  Annees : 1896 - 2022

populations.csv :
  Annees : 1960 - 2023


In [25]:
# === Verification : Paris 2024 est-il present ? ===
# Critique pour le projet "JO 2028" : on a besoin de la baseline 2024
# comme reference la plus recente pour les analyses et le modele ML.

print("=== Verification Paris 2024 ===")

# Dans results.csv (KeithGalli) : couverture jusqu'a 2022 -> Paris 2024 absent
# MODIF : filtre sur "year" au lieu de "Games" (qui n'existe pas dans KeithGalli)
paris_results = results[results["year"] == 2024]
print(f"\nresults.csv -- participations 2024 : {len(paris_results):,}")
if len(paris_results) > 0:
    print(f"  Disciplines : {paris_results['discipline'].nunique()}")
    print(f"  Medailles : {paris_results['medal'].notna().sum()}")
else:
    # Cas attendu avec KeithGalli (snapshot 2022)
    print("  Paris 2024 absent de results.csv (couverture jusqu'a 2022)")
    print("  -> on utilisera olympics_1896_2024.csv pour les donnees 2024")

# olympics_1896_2024.csv : couvre Paris 2024, integre en phase 3 (nettoyage)
paris_olympics = olympics[olympics["event_year"] == 2024]
print(f"\nolympics_1896_2024.csv -- medailles 2024 : {len(paris_olympics):,}")
if len(paris_olympics) > 0:
    print(f"  Ville : {paris_olympics['city'].unique()}")
    print(f"  Disciplines : {paris_olympics['discipline'].nunique()}")


=== Verification Paris 2024 ===

results.csv -- participations 2024 : 0
  Paris 2024 absent de results.csv (couverture jusqu'a 2022)
  -> on utilisera olympics_1896_2024.csv pour les donnees 2024

olympics_1896_2024.csv -- medailles 2024 : 2,315
  Ville : <ArrowStringArray>
['Paris']
Length: 1, dtype: str
  Disciplines : 45


In [26]:
# MODIF : avant, cette cellule supprimait la colonne 'year' temporaire creee
# par extraction de 'Games'. Avec KeithGalli, 'year' est native dans results.csv,
# pas besoin de la supprimer.
print(f"results.csv contient {len(results):,} lignes apres verifications")


results.csv contient 308,408 lignes apres verifications


### Coherence entre `olympics_1896_2024` et `city_locations`

Verifions que toutes les villes du dataset de medailles ont des coordonnees.

In [27]:
olympics_cities = set(olympics["city"].dropna().unique())
ref_cities = set(city_locations["City"].dropna().unique())

missing_cities = olympics_cities - ref_cities

print(f"Villes dans olympics_1896_2024 : {len(olympics_cities)}")
print(f"Villes dans city_locations :     {len(ref_cities)}")
print(f"\nVilles sans coordonnees ({len(missing_cities)}) :")
if missing_cities:
    for city in sorted(missing_cities):
        print(f"  {city}")
else:
    print("  Aucune -- toutes les villes ont des coordonnees.")

Villes dans olympics_1896_2024 : 43
Villes dans city_locations :     44

Villes sans coordonnees (0) :
  Aucune -- toutes les villes ont des coordonnees.


---

## Conclusion

### Donnees acquises

Nous disposons de **7 datasets** couvrant l'ensemble de l'histoire olympique,
de 1896 (Athenes) a 2024 (Paris) :

| Dataset | Contenu | Volume |
|---------|---------|--------|
| `bios.csv` | Biographies des athletes | ~145 500 athletes |
| `bios_locs.csv` | Biographies + geolocalisation | ~145 500 athletes |
| `results.csv` | Participations et resultats | ~308 400 participations |
| `olympics_1896_2024.csv` | Medailles consolidees | ~24 000 medailles |
| `noc_regions.csv` | Mapping NOC -> pays | 229 codes |
| `populations.csv` | Population par pays (1960-2023) | 266 entites |
| `city_locations.csv` | Coordonnees des villes hotes | 55 editions |

### Points d'attention pour le nettoyage

Plusieurs chantiers se dessinent pour le prochain notebook :

1. **Valeurs manquantes** -- Les colonnes biographiques (`Died`, `Nick/petnames`, `Measurements`) ont des taux de NaN eleves. Pour les mensurations (`height_cm`, `weight_kg`), une strategie d'imputation ou de filtrage sera necessaire.

2. **Colonne parasite** -- `Unnamed: 7` dans `results.csv` est a supprimer.

3. **Harmonisation des codes pays** -- Les codes NOC (results) et ISO (populations) ne sont pas identiques. Une table de correspondance devra etre construite.

4. **Parsing des dates** -- La colonne `Born` de `bios.csv` est en texte libre ; `bios_locs.csv` offre une alternative deja structuree.

5. **Doublons potentiels** -- A verifier dans `results.csv` (un meme athlete pourrait apparaitre plusieurs fois pour la meme epreuve).

6. **Coherence des noms** -- Les noms d'athletes et de pays peuvent differer entre `results.csv` et `olympics_1896_2024.csv` (sources differentes).

### Sources completes

| Source | URL |
|--------|-----|
| KeithGalli/Olympics-Dataset | https://github.com/KeithGalli/Olympics-Dataset |
| KeithGalli/Olympics-Dataset (clean-data) | https://github.com/KeithGalli/Olympics-Dataset/tree/main/clean-data |
| DjangoMustang/Olympics-1896-2024-Tableau | https://github.com/DjangoMustang/Olympics-1896-2024-Tableau |
| Banque Mondiale (via KeithGalli) | https://data.worldbank.org/indicator/SP.POP.TOTL |